# Three-seed GPU retraining and matched evaluation

This is the next stage after the verified 90-run retention experiment. Open in
VS Code using the existing Kaggle GPU Jupyter kernel. Enable GPU and Internet.
All three seed blocks below are **enabled**: 42, 73, 101, each training LiteCrack,
LiteCrack-BA and stabilized R2AU from scratch. No parent model attachment is needed
for source retraining; the exact audited source/configs are embedded.

New directory: `/kaggle/working/road_crack_multiseed_v1`. Previous experiments stay
unchanged. A separate recovery ZIP and SHA256 are written after every seed block
and evaluation/adaptation stage, including a safely stopped stage. Download them
after each block. GPU time and disk use are substantial; stop between blocks when
needed. Source checkpoints use the frozen 30-step/epoch cadence. On low storage,
restore a recovery ZIP in a fresh session; do not delete original snapshots blindly.

For a resumed session attach the latest **multiseed_*.zip**, or its complete
extracted contents. Restore supports both forms and verifies the full inventory.
Exact resume requires the original compatible runtime. This notebook does not
fabricate independent road scenes or timed human corrections.


In [ ]:
payload='...'
payload_sha256='3824768de9b43f127878d4fca7ace4bd4f56c3ab740dc2b15554ada80162704c'
import base64,zlib,json,hashlib,sys,os,subprocess,importlib.util
from pathlib import Path
raw=zlib.decompress(base64.b64decode(payload));assert hashlib.sha256(raw).hexdigest()==payload_sha256
files=json.loads(raw);expected_plan=json.loads(files['MULTISEED_PLAN.json'])
project=Path('/kaggle/working/road_crack_multiseed_v1');project.mkdir(exist_ok=True)
had_state=(project/'MULTISEED_PLAN.json').exists() or (project/'source_multiseed').exists()
for name,text in files.items():
    if name=='MULTISEED_PLAN.json':continue
    target=project/name;target.parent.mkdir(parents=True,exist_ok=True)
    content=text.encode('utf-8')
    if target.exists():assert target.read_bytes()==content,'Existing code differs; preserve the old project'
    else:target.write_bytes(content)
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
spec=importlib.util.spec_from_file_location('multiseed_bootstrap',project/'scripts/multiseed_pipeline.py')
helper=importlib.util.module_from_spec(spec);spec.loader.exec_module(helper)
if not had_state:
    candidates=list(Path('/kaggle/input').rglob('multiseed_*.zip'))
    candidates+=[p.parent for p in Path('/kaggle/input').rglob('MULTISEED_PLAN.json') if (p.parent/'FOLLOWUP_BACKUP_MANIFEST.json').is_file()]
    backup=helper.latest_backup(candidates,expected_plan)
    if backup:print(helper.restore(backup,project,expected_plan))
    else:print('Starting a new three-seed study; no matching recovery input found.')
plan=project/'MULTISEED_PLAN.json'
if plan.exists():assert json.loads(plan.read_text())==expected_plan,'Protocol changed'
else:plan.write_bytes(files['MULTISEED_PLAN.json'].encode())
helper.verify_project(project)
import torch
assert torch.cuda.is_available(),'GPU required: enable Kaggle GPU and reconnect the existing Jupyter kernel'
print('GPU:',torch.cuda.get_device_name(0),'PyTorch:',torch.__version__)
os.chdir(project)
subprocess.run([sys.executable,'scripts/restore_datasets.py','--manifest','data/manifests/pilot_v1.json',
    '--destination','/kaggle/working/road_crack_followup_data','--sources','configs/data_sources.json',
    '--report','dataset_restore.json'],check=True)
data_root=Path(json.loads((project/'dataset_restore.json').read_text())['data_root'])
with (project/'gpu-environment.lock.txt').open('w') as stream:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=stream,check=True)
backup_dir=Path('/kaggle/working')


In [ ]:
import xml.etree.ElementTree as ET
from uuid import uuid4
evidence=project/'test_evidence';evidence.mkdir(exist_ok=True)
result=subprocess.run([sys.executable,'-m','pytest','tests/test_followup.py','tests/test_phase2.py','tests/test_multiseed_pipeline.py',
    '-q','-rA','-p','no:cacheprovider','--basetemp='+str(evidence/('tmp-'+uuid4().hex)),
    '--junitxml='+str(evidence/'tests.xml')],capture_output=True,text=True)
(evidence/'tests.log').write_text(result.stdout+result.stderr)
print(result.stdout,result.stderr);assert result.returncode==0,'Fix checks before training'
cases=ET.parse(evidence/'tests.xml').findall('.//testcase')
recovery=[c for c in cases if any(n in c.attrib.get('name','') for n in ('test_actual_trainer_mid_epoch_resume','test_followup_exact_resume'))]
assert len(recovery)==12 and all(c.find('skipped') is None for c in recovery),'CPU and CUDA recovery must pass for source training and adaptation'


In [ ]:
def run_stage(stage,seed=None):
    command=[sys.executable,'scripts/multiseed_pipeline.py','--project',str(project),'--data-root',str(data_root),
        '--stage',stage,'--backup-dir',str(backup_dir)]
    if seed is not None:command+=['--seed',str(seed)]
    import signal
    worker=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in worker.stdout:print(line,end='',flush=True)
        status=worker.wait()
    except KeyboardInterrupt:
        # SIGINT is handled by the orchestrator, which waits for its training worker.
        worker.send_signal(signal.SIGINT);worker.wait(timeout=180);raise
    assert status==0,'Stage stopped or failed; download recovery, then rerun the same cell to resume'
print('Three seeds are enabled. Run the next cells in order; download each printed recovery ZIP.')


In [ ]:
# Enabled source-training block: three models, seed 42.
run_stage('train',42)


RECOVERY ZIP: /kaggle/working/multiseed_seed42_20261101T141156391873Z.zip
SHA256: 60f13ff64b75c63926db074b54b3bf3101bd0e9e3828bf1befc2b2f60263757b


In [ ]:
from pathlib import Path
import shutil

fixture = project / "test_evidence" / "tmp-8ac8578977fb41358fa2962a4da7209f"
quarantine = Path("/kaggle/working/road_crack_multiseed_test_fixture_quarantine") / fixture.name

assert fixture.is_dir(), f"Fixture directory not found: {fixture}"
assert fixture.resolve().is_relative_to(project.resolve())
assert not quarantine.exists(), f"Quarantine destination already exists: {quarantine}"
quarantine.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(fixture), str(quarantine))
print("Preserved test fixture at:", quarantine)

Preserved test fixture at: /kaggle/working/road_crack_multiseed_test_fixture_quarantine/tmp-8ac8578977fb41358fa2962a4da7209f


In [ ]:
# Enabled source-training block: three models, seed 73.
run_stage('train',73)


{"epoch": 1, "step": 30, "train_loss": 1.805368435382843, "validation": {"loss": 0.9404663403828939, "dice": 0.10098494946335752, "iou": 0.054352857545018195, "precision": 0.055781682347878815, "recall": 0.7306335046887398, "boundary_f1": 0.14859209787100552, "predicted_foreground_fraction": 0.37157313028971356, "truth_foreground_fraction": 0.028463999430338543, "n": 60}, "learning_rate": 0.0001}
{"epoch": 2, "step": 60, "train_loss": 1.6802357037862141, "validation": {"loss": 0.9438284794489543, "dice": 0.1501897885153691, "iou": 0.08462986412923783, "precision": 0.09252418681668738, "recall": 0.5403424854079882, "boundary_f1": 0.295691302853326, "predicted_foreground_fraction": 0.169891357421875, "truth_foreground_fraction": 0.028463999430338543, "n": 60}, "learning_rate": 0.0001}
{"epoch": 3, "step": 90, "train_loss": 1.6367169141769409, "validation": {"loss": 0.9455706834793091, "dice": 0.17749293589343626, "iou": 0.1023250986666729, "precision": 0.12008716117901107, "recall": 0.43

In [ ]:
# Enabled source-training block: three models, seed 101.
run_stage('train',101)


In [ ]:
# Requires all nine source runs; no best-seed selection.
run_stage('evaluate')


In [ ]:
# Six LiteCrack/BA parents, validation-only selection then locked adaptation.
# Download recovery and restore in fresh storage if disk headroom is exhausted.
run_stage('adapt')


The final `multiseed_matrix/summary.json` separates training-seed SD from
support-draw SD. Completed parents with no admissible candidate remain recorded
as failures; no gate is relaxed. Historical query reuse remains exploratory.
New site-separated scene annotations and genuine human timing sessions are the
remaining external inputs. See the local `docs/NEXT_GPU_RUN.md` guide.
